# Эксперименты
<code>Эксперимент 1</code>: Влияние активационной функции на сходимость
- Цель эксперимента -> Понять, как выбор функции активации (<code>LeakyReLU</code> vs <code>ReLU</code>) влияет на скорость сходимости и результаты обучения. 
- Идея эксперимента -> Заменить LeakyReLU на ReLU и провести аналогичные эксперименты.
- Результаты эксперимента -> Графики train/test loss.
- Сравнение с baseline: Сходимость с LeakyReLU происходит давольно быстрее, посольку все нейроны, которые имеют отрицательные градиенты также влияют на направление сходимости, и не зануляются как в случае с ReLU.
- Выводы -> Использование LeakyReLU помогает быстрее достигать сходимости при минимальной разнице в точности (на ~1%).

<code>Эксперимент 2</code>: Влияние регуляризации на сходимость и точность модели
- Цель эксперимента -> Проверить, сохранит ли модель аналогичную или близкую точность, а также как изменится процесс обучения по сравнению с базовой версией, если использовать Dropout.
- Идея эксперимента -> Обучить модель на 10 эпох и сравнить графики train/test loss c безлайном. Если разрыв между train/test увеличивается, это может свидетельствовать о переобучении, что прослеживалось и в безлайне.
- Результаты эксперимента -> Графики train/test loss.
- Сравнение с baseline -> Отключенные нейроны во время обучения повышают обобщающую способность используемых нейронов, но сильно влияют на скорость сходимости в целом, что и влияет на точность модели.
- Выводы -> Добавление Dropout помогает снизить переобучение, что подтверждается меньшим разрывом между train/test loss, однако из-за сходимости модель теряется в точности.

### Эксперимент 1

In [15]:
import torch
import wandb
from tqdm import tqdm
import torchvision as tv
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.cuda.amp import autocast
import torch.optim as optim
import torch.nn.functional as F
from torchvision.datasets import CIFAR10
import torchvision.transforms as transforms
from torchvision.utils import make_grid
from collections import defaultdict
from torch.utils.data.dataloader import DataLoader
from torch.utils.data import random_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score



import warnings
warnings.filterwarnings('ignore')

### Загружаем данные

In [16]:
transform = transforms.Compose([transforms.ToTensor(),
                                 transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 16
trainset = tv.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = tv.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat','deer', 'dog', 'frog', 'horse', 'ship', 'truck')

Files already downloaded and verified
Files already downloaded and verified


### Метрики подсчета

In [17]:
def count_parameteres(model):
    """ Считаем парметры модели """
    count_parm = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Количество параметров модели: {count_parm}')

def accuracy(pred, label):
    answer = F.softmax(pred.detach()).numpy().argmax(1) == label.numpy() 
    return answer.mean()

def evaluate(model, dataloader, device, loss_fn, use_amp):
    model.eval()
    loss_val = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for img, label in dataloader:
            img, label = img.to(device), label.to(device)
            with torch.cuda.amp.autocast(enabled=use_amp):
                pred = model(img)
                loss = loss_fn(pred, label)
            
            loss_val += loss.item()
            all_preds.append(pred.argmax(dim=1).cpu().numpy())
            all_labels.append(label.cpu().numpy())

    avg_loss = loss_val / len(dataloader)
    
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    precision = precision_score(all_labels, all_preds, average='macro')
    recall = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')
    accuracy = (all_preds == all_labels).mean()
    return avg_loss, accuracy, precision, recall, f1

### Модель

In [18]:
class ResBlock(nn.Module):
    def __init__(self, num_ch):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(num_ch, num_ch, kernel_size=3, padding=1)
        self.norm1 = nn.BatchNorm2d(num_ch)
        self.act = nn.LeakyReLU(0.2)
        self.conv2 = nn.Conv2d(num_ch, num_ch,kernel_size=3, padding=1)
        self.norm2 = nn.BatchNorm2d(num_ch)
    
    def forward(self, x):
        out = self.conv1(x)
        out = self.norm1(out)
        out = self.act(out)        
        out = self.conv2(out)
        out = self.norm2(out)

        return self.act(x + out)


class ResTruck(nn.Module):
    def __init__(self, num_ch, num_blocks):
        super(ResTruck, self).__init__()
        
        truck = []        
        for i in range(num_blocks):
            truck.append(ResBlock(num_ch))
        self.truck = nn.Sequential(*truck)
            
    def forward(self, x):
        return self.truck(x)


class ResNet(nn.Module):
    def __init__(self, in_ch, num_ch, out_ch):
        super(ResNet, self).__init__()
        
        self.conv0 = nn.Conv2d(in_ch, num_ch, kernel_size=7, stride=2, padding=3)
        self.norm = nn.BatchNorm2d(num_ch)
        self.act = nn.LeakyReLU(0.2)
        self.maxpool = nn.MaxPool2d(2,2, padding=1)

        self.layer1 = ResTruck(num_ch, 3) 
        self.conv1 = nn.Conv2d(num_ch, 2*num_ch, 3, padding=1, stride=2)
        
        self.layer2 = ResTruck(2*num_ch, 4) 
        self.conv2 = nn.Conv2d(2*num_ch, 4*num_ch, 3, padding=1, stride=2)
        
        self.layer3 = ResTruck(4*num_ch, 6) 
        self.conv3 = nn.Conv2d(4*num_ch, 8*num_ch, 3, padding=1, stride=2)
        
        self.layer4 = ResTruck(8*num_ch, 3) 
        
        self.avgpool = nn.AdaptiveAvgPool2d(output_size=(1,1))
        self.linear = nn.Linear(in_features=8*num_ch, out_features=out_ch)
    
    def forward(self, x):
        x = self.conv0(x)
        # x = self.norm(x)
        x = self.act(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.conv1(x)
        x = self.layer2(x)
        x = self.conv2(x)
        x = self.layer3(x)
        x = self.conv3(x)
        x = self.layer4(x)
        x = self.avgpool(x)        
        x = self.linear(torch.flatten(x, 1))

        return x

In [19]:
import wandb
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ResNet(3, 64, 10).to(device)
loss_fn = nn.CrossEntropyLoss().to(device)

wandb.init(
    project="resnet34",
    config={
        "learning_rate": 0.001,
        "epochs": 10,
        "batch_size": trainloader.batch_size,
        "optimizer": "AdamW",
        "scheduler": "ExponentialLR",
        "model": "ResNet",
    },
)
wandb.watch(model, log="all", log_freq=10)


optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, betas=(0.8, 0.999))
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.6)

use_amp = True
scaler = torch.cuda.amp.GradScaler()
torch.backends.cudnn.benchmark = True

epochs = 10
train_loss = []
test_loss = []

acc_train_list = []
acc_test_list = []
precision_list = []
recall_list = []
f1_list = []

for epoch in range(epochs):
    model.train()
    loss_val = 0
    acc_val = 0
    for sample in tqdm(trainloader, desc=f"Epoch {epoch + 1}/{epochs}"):
        img, label = sample
        img, label = img.to(device), label.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            pred = model(img)
            loss = loss_fn(pred, label)

        scaler.scale(loss).backward()
        loss_item = loss.item()
        loss_val += loss_item

        scaler.step(optimizer)
        scaler.update()

        acc_current = accuracy(pred.cpu().float(), label.cpu().float())
        acc_val += acc_current

    scheduler.step()

    train_loss_epoch = loss_val / len(trainloader)
    train_acc_epoch = acc_val / len(trainloader)
    train_loss.append(train_loss_epoch)
    acc_train_list.append(train_acc_epoch)

    test_loss_epoch, test_acc_epoch, precision_epoch, recall_epoch, f1_epoch = evaluate(
        model, testloader, device, loss_fn, use_amp
    )
    test_loss.append(test_loss_epoch)
    acc_test_list.append(test_acc_epoch)
    precision_list.append(precision_epoch)
    recall_list.append(recall_epoch)
    f1_list.append(f1_epoch)

    wandb.log({
        "train_loss": train_loss_epoch,
        "test_loss": test_loss_epoch,
        "train_accuracy": train_acc_epoch,
        "test_accuracy": test_acc_epoch,
        "precision": precision_epoch,
        "recall": recall_epoch,
        "f1_score": f1_epoch,        
    })

    print(f'Epoch: {epoch + 1}/{epochs}')
    print(f'Train Loss: {train_loss_epoch:.4f} | Train Accuracy: {train_acc_epoch:.4f}')
    print(f'Test Loss: {test_loss_epoch:.4f} | Test Accuracy: {test_acc_epoch:.4f}')
    print(f'Precision: {precision_epoch:.4f} | Recall: {recall_epoch:.4f} | F1-Score: {f1_epoch:.4f}')


wandb.finish()

Epoch 1/10: 100%|███████████████████████████████████████| 3125/3125 [02:10<00:00, 23.99it/s]


Epoch: 1/10
Train Loss: 1.5598 | Train Accuracy: 0.4265
Test Loss: 1.3090 | Test Accuracy: 0.5212
Precision: 0.5527 | Recall: 0.5212 | F1-Score: 0.5104


Epoch 2/10: 100%|███████████████████████████████████████| 3125/3125 [02:23<00:00, 21.72it/s]


Epoch: 2/10
Train Loss: 1.0876 | Train Accuracy: 0.6099
Test Loss: 1.0146 | Test Accuracy: 0.6411
Precision: 0.6464 | Recall: 0.6411 | F1-Score: 0.6387


Epoch 3/10: 100%|███████████████████████████████████████| 3125/3125 [02:13<00:00, 23.44it/s]


...

Epoch 9/10: 100%|███████████████████████████████████████| 3125/3125 [02:30<00:00, 20.73it/s]


Epoch: 9/10
Train Loss: 0.2573 | Train Accuracy: 0.9153
Test Loss: 0.8667 | Test Accuracy: 0.7506
Precision: 0.7498 | Recall: 0.7506 | F1-Score: 0.7500


Epoch 10/10: 100%|██████████████████████████████████████| 3125/3125 [02:07<00:00, 24.42it/s]


Epoch: 10/10
Train Loss: 0.2300 | Train Accuracy: 0.9258
Test Loss: 0.8906 | Test Accuracy: 0.7503
Precision: 0.7504 | Recall: 0.7503 | F1-Score: 0.7502


f1_score,▁▅▆▇██████
precision,▁▄▆▇██████
recall,▁▅▆▇██████
test_accuracy,▁▅▆▇██████
test_loss,█▄▂▁▁▁▂▂▂▃
train_accuracy,▁▄▅▆▆▇▇███
train_loss,█▆▄▃▃▂▂▁▁▁
f1_score,0.75017
precision,0.75037
recall,0.7503
test_accuracy,0.7503


In [24]:
import wandb

wandb.init(mode="disabled")

correct_per_class = defaultdict(int)  
total_per_class = defaultdict(int)    

model.eval()
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        for label, pred in zip(labels, predicted):
            total_per_class[label.item()] += 1
            if label.item() == pred.item():
                correct_per_class[label.item()] += 1

per_class_accuracy = {}
for cls_idx in range(len(classes)):
    accuracy = correct_per_class[cls_idx] / total_per_class[cls_idx] if total_per_class[cls_idx] > 0 else 0
    per_class_accuracy[classes[cls_idx]] = accuracy

for cls_name, acc in per_class_accuracy.items():
    print(f"{cls_name}: {acc * 100:.2f}%")

plane: 82.00%
car: 85.60%
bird: 65.70%
cat: 54.10%
deer: 72.20%
dog: 63.00%
frog: 81.60%
horse: 79.60%
ship: 85.20%
truck: 80.90%


### Эксперимент 2

In [31]:
import torch.nn.functional as F

class ResBlock(nn.Module):
    def __init__(self, num_ch, dropout_rate=0.3): 
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(num_ch, num_ch, kernel_size=3, padding=1)
        self.norm1 = nn.BatchNorm2d(num_ch)
        self.act = nn.LeakyReLU(0.2)
        self.conv2 = nn.Conv2d(num_ch, num_ch, kernel_size=3, padding=1)
        self.norm2 = nn.BatchNorm2d(num_ch)
        self.dropout = nn.Dropout2d(dropout_rate) 

    def forward(self, x):
        out = self.conv1(x)
        out = self.norm1(out)
        out = self.act(out)
        out = self.dropout(out) 
        out = self.conv2(out)
        out = self.norm2(out)
        out = self.dropout(out) 
        return self.act(x + out)


class ResTruck(nn.Module):
    def __init__(self, num_ch, num_blocks, dropout_rate=0.3):
        super(ResTruck, self).__init__()
        
        truck = []
        for i in range(num_blocks):
            truck.append(ResBlock(num_ch, dropout_rate))
        self.truck = nn.Sequential(*truck)
            
    def forward(self, x):
        return self.truck(x)


class ResNet(nn.Module):
    def __init__(self, in_ch:int, num_ch:int, out_ch:int, dropout_rate=0.3, weight_decay=1e-4):
        super(ResNet, self).__init__()
        
        self.conv0 = nn.Conv2d(in_ch, num_ch, kernel_size=7, stride=2, padding=3)
        self.norm = nn.BatchNorm2d(num_ch)
        self.act = nn.LeakyReLU(0.2)
        self.maxpool = nn.MaxPool2d(2,2, padding=1)

        self.layer1 = ResTruck(num_ch, 3, dropout_rate)
        self.conv1 = nn.Conv2d(num_ch, 2*num_ch, 3, padding=1, stride=2)
        
        self.layer2 = ResTruck(2*num_ch, 4, dropout_rate)
        self.conv2 = nn.Conv2d(2*num_ch, 4*num_ch, 3, padding=1, stride=2)
        
        self.layer3 = ResTruck(4*num_ch, 6, dropout_rate)
        self.conv3 = nn.Conv2d(4*num_ch, 8*num_ch, 3, padding=1, stride=2)
        
        self.layer4 = ResTruck(8*num_ch, 3, dropout_rate)
        
        self.avgpool = nn.AdaptiveAvgPool2d(output_size=(1,1))
        self.linear = nn.Linear(in_features=8*num_ch, out_features=out_ch)
        
        self.dropout = nn.Dropout(dropout_rate)  

    def forward(self, x):
        x = self.conv0(x)
        x = self.act(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.conv1(x)
        x = self.layer2(x)
        x = self.conv2(x)
        x = self.layer3(x)
        x = self.conv3(x)
        x = self.layer4(x)
        x = self.avgpool(x)        
        x = self.linear(torch.flatten(x, 1))
        x = self.dropout(x) 

        return x

In [33]:
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ResNet(in_ch=3, num_ch=64, out_ch=10, dropout_rate=0.3, weight_decay=1e-4).to(device)
loss_fn = nn.CrossEntropyLoss().to(device)


def accuracy(pred, label):
    answer = F.softmax(pred.detach()).numpy().argmax(1) == label.numpy() 
    return answer.mean()

def evaluate(model, dataloader, device, loss_fn, use_amp):
    model.eval()
    loss_val = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for img, label in dataloader:
            img, label = img.to(device), label.to(device)
            with torch.cuda.amp.autocast(enabled=use_amp):
                pred = model(img)
                loss = loss_fn(pred, label)
            
            loss_val += loss.item()
            all_preds.append(pred.argmax(dim=1).cpu().numpy())
            all_labels.append(label.cpu().numpy())

    avg_loss = loss_val / len(dataloader)
    
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    precision = precision_score(all_labels, all_preds, average='macro')
    recall = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')
    accuracy = (all_preds == all_labels).mean()
    return avg_loss, accuracy, precision, recall, f1

wandb.init(
    project="resnet34",
    config={
        "learning_rate": 0.001,
        "epochs": 10,
        "batch_size": trainloader.batch_size,
        "optimizer": "AdamW",
        "scheduler": "ExponentialLR",
        "model": "ResNet",
    },
)
wandb.watch(model, log="all", log_freq=10)


optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, betas=(0.8, 0.999))
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.6)

use_amp = True
scaler = torch.cuda.amp.GradScaler()
torch.backends.cudnn.benchmark = True

epochs = 10
train_loss = []
test_loss = []

acc_train_list = []
acc_test_list = []
precision_list = []
recall_list = []
f1_list = []

for epoch in range(epochs):
    model.train()
    loss_val = 0
    acc_val = 0
    for sample in tqdm(trainloader, desc=f"Epoch {epoch + 1}/{epochs}"):
        img, label = sample
        img, label = img.to(device), label.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            pred = model(img)
            loss = loss_fn(pred, label)

        scaler.scale(loss).backward()
        loss_item = loss.item()
        loss_val += loss_item

        scaler.step(optimizer)
        scaler.update()

        acc_current = accuracy(pred.cpu().float(), label.cpu().float())
        acc_val += acc_current

    scheduler.step()

    train_loss_epoch = loss_val / len(trainloader)
    train_acc_epoch = acc_val / len(trainloader)
    train_loss.append(train_loss_epoch)
    acc_train_list.append(train_acc_epoch)

    test_loss_epoch, test_acc_epoch, precision_epoch, recall_epoch, f1_epoch = evaluate(
        model, testloader, device, loss_fn, use_amp
    )
    test_loss.append(test_loss_epoch)
    acc_test_list.append(test_acc_epoch)
    precision_list.append(precision_epoch)
    recall_list.append(recall_epoch)
    f1_list.append(f1_epoch)

    wandb.log({
        "train_loss": train_loss_epoch,
        "test_loss": test_loss_epoch,
        "train_accuracy": train_acc_epoch,
        "test_accuracy": test_acc_epoch,
        "precision": precision_epoch,
        "recall": recall_epoch,
        "f1_score": f1_epoch,        
    })

    print(f'Epoch: {epoch + 1}/{epochs}')
    print(f'Train Loss: {train_loss_epoch:.4f} | Train Accuracy: {train_acc_epoch:.4f}')
    print(f'Test Loss: {test_loss_epoch:.4f} | Test Accuracy: {test_acc_epoch:.4f}')
    print(f'Precision: {precision_epoch:.4f} | Recall: {recall_epoch:.4f} | F1-Score: {f1_epoch:.4f}')


wandb.finish()

Epoch 1/10: 100%|███████████████████████████████████████| 3125/3125 [02:15<00:00, 23.06it/s]


Epoch: 1/10
Train Loss: 1.9311 | Train Accuracy: 0.2974
Test Loss: 1.5154 | Test Accuracy: 0.4472
Precision: 0.4574 | Recall: 0.4472 | F1-Score: 0.4221


Epoch 2/10: 100%|███████████████████████████████████████| 3125/3125 [02:37<00:00, 19.89it/s]


Epoch: 2/10
Train Loss: 1.6182 | Train Accuracy: 0.4165
Test Loss: 1.2772 | Test Accuracy: 0.5494
Precision: 0.5673 | Recall: 0.5494 | F1-Score: 0.5423


Epoch 3/10: 100%|███████████████████████████████████████| 3125/3125 [02:18<00:00, 22.63it/s]


...

Epoch 9/10: 100%|███████████████████████████████████████| 3125/3125 [02:22<00:00, 21.97it/s]


Epoch: 9/10
Train Loss: 1.1299 | Train Accuracy: 0.5758
Test Loss: 0.8870 | Test Accuracy: 0.6920
Precision: 0.6890 | Recall: 0.6920 | F1-Score: 0.6890


Epoch 10/10: 100%|██████████████████████████████████████| 3125/3125 [02:32<00:00, 20.52it/s]


Epoch: 10/10
Train Loss: 1.1217 | Train Accuracy: 0.5794
Test Loss: 0.8755 | Test Accuracy: 0.6949
Precision: 0.6918 | Recall: 0.6949 | F1-Score: 0.6927


f1_score,▁▄▆▇▇█████
precision,▁▄▆▇▇█████
recall,▁▄▅▇▇█████
test_accuracy,▁▄▅▇▇█████
test_loss,█▅▄▃▂▂▁▁▁▁
train_accuracy,▁▄▅▆▇▇████
train_loss,█▅▄▃▂▂▁▁▁▁
f1_score,0.69269
precision,0.69183
recall,0.6949
test_accuracy,0.6949


In [35]:
wandb.init(mode="disabled")

correct_per_class = defaultdict(int)  
total_per_class = defaultdict(int)    

model.eval()
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        for label, pred in zip(labels, predicted):
            total_per_class[label.item()] += 1
            if label.item() == pred.item():
                correct_per_class[label.item()] += 1

per_class_accuracy = {}
for cls_idx in range(len(classes)):
    accuracy = correct_per_class[cls_idx] / total_per_class[cls_idx] if total_per_class[cls_idx] > 0 else 0
    per_class_accuracy[classes[cls_idx]] = accuracy

for cls_name, acc in per_class_accuracy.items():
    print(f"{cls_name}: {acc * 100:.2f}%")

plane: 76.40%
car: 82.30%
bird: 57.10%
cat: 49.90%
deer: 58.20%
dog: 57.60%
frog: 80.10%
horse: 76.90%
ship: 82.50%
truck: 73.70%
